In [5]:
pip install opencv-python dlib imutils scipy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import cv2
import dlib
import time
from scipy.spatial import distance


In [7]:
import cv2
import dlib
import imutils

from pygame import mixer
from scipy.spatial import distance
from imutils import face_utils
from collections import deque


# ============================================================
# 1. INITIALIZE AUDIO
# ============================================================

mixer.init()

# Keep alert.mp3 in the same folder as this Python file
mixer.music.load("alert.mp3")


# ============================================================
# 2. EYE ASPECT RATIO (EAR)
# ============================================================

def ear(eye):
    """
    Calculate Eye Aspect Ratio.

    Higher EAR = eyes open
    Lower EAR  = eyes closed
    """

    A = distance.euclidean(eye[1], eye[5])
    B = distance.euclidean(eye[2], eye[4])
    C = distance.euclidean(eye[0], eye[3])

    return (A + B) / (2.0 * C)


# ============================================================
# 3. MOUTH ASPECT RATIO (MAR)
# ============================================================

def mar(mouth):
    """
    Calculate Mouth Aspect Ratio.

    Higher MAR = mouth open
    """

    A = distance.euclidean(mouth[13], mouth[19])
    B = distance.euclidean(mouth[14], mouth[18])
    C = distance.euclidean(mouth[15], mouth[17])
    D = distance.euclidean(mouth[12], mouth[16])

    return (A + B + C) / (3.0 * D)


# ============================================================
# 4. SETTINGS / THRESHOLDS
# ============================================================

# Based on your EAR values
EYE_THRESH = 0.18

# Eyes must remain closed for this many frames
EYE_CONSEC_FRAMES = 15

# Mouth threshold for yawning
MOUTH_THRESH = 0.70

# Mouth must remain open for this many frames
MOUTH_CONSEC_FRAMES = 15


# ============================================================
# 5. COUNTERS
# ============================================================

eye_counter = 0
mouth_counter = 0


# ============================================================
# 6. ALERT STATES
# ============================================================

drowsy_alert_played = False
yawn_alert_played = False


# ============================================================
# 7. EAR SMOOTHING
# ============================================================

# Stores the last 5 EAR values
ear_history = deque(maxlen=5)


# ============================================================
# 8. FACE DETECTOR
# ============================================================

detector = dlib.get_frontal_face_detector()


# ============================================================
# 9. LOAD 68-POINT FACE LANDMARK MODEL
# ============================================================

predictor_path = (
    "C:/Users/polep/shape_predictor_68_face_landmarks.dat"
)

predictor = dlib.shape_predictor(predictor_path)


# ============================================================
# 10. FACIAL LANDMARK INDEXES
# ============================================================

(left_start, left_end) = (
    face_utils.FACIAL_LANDMARKS_68_IDXS["left_eye"]
)

(right_start, right_end) = (
    face_utils.FACIAL_LANDMARKS_68_IDXS["right_eye"]
)

(mouth_start, mouth_end) = (
    face_utils.FACIAL_LANDMARKS_68_IDXS["mouth"]
)


# ============================================================
# 11. START WEBCAM
# ============================================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():

    print("ERROR: Could not open webcam.")
    exit()


print("========================================")
print("DROWSINESS & YAWN DETECTION STARTED")
print("========================================")
print("Press Q to quit.")


# ============================================================
# 12. MAIN LOOP
# ============================================================

while True:

    ret, frame = cap.read()

    if not ret:

        print("ERROR: Could not read webcam.")
        break


    # --------------------------------------------------------
    # Mirror webcam
    # --------------------------------------------------------

    frame = cv2.flip(frame, 1)


    # --------------------------------------------------------
    # Resize frame
    # --------------------------------------------------------

    frame = imutils.resize(frame, width=600)


    # --------------------------------------------------------
    # Convert to grayscale
    # --------------------------------------------------------

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)


    # --------------------------------------------------------
    # Detect faces
    # --------------------------------------------------------

    faces = detector(gray)


    # ========================================================
    # NO FACE DETECTED
    # ========================================================

    if len(faces) == 0:

        eye_counter = 0
        mouth_counter = 0

        ear_history.clear()

        # Stop any alert if face disappears
        mixer.music.stop()

        drowsy_alert_played = False
        yawn_alert_played = False

        cv2.putText(
            frame,
            "NO FACE DETECTED",
            (10, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

        cv2.imshow(
            "Drowsiness & Yawn Detection",
            frame
        )

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

        continue


    # ========================================================
    # SELECT LARGEST FACE
    # ========================================================

    face = max(
        faces,
        key=lambda rect: rect.width() * rect.height()
    )


    # ========================================================
    # GET 68 FACIAL LANDMARKS
    # ========================================================

    shape = predictor(gray, face)

    shape = face_utils.shape_to_np(shape)


    # ========================================================
    # GET EYE LANDMARKS
    # ========================================================

    left_eye = shape[left_start:left_end]

    right_eye = shape[right_start:right_end]


    # ========================================================
    # CALCULATE EAR
    # ========================================================

    left_ear = ear(left_eye)

    right_ear = ear(right_eye)

    avg_ear = (left_ear + right_ear) / 2.0


    # ========================================================
    # SMOOTH EAR
    # ========================================================

    ear_history.append(avg_ear)

    smooth_ear = sum(ear_history) / len(ear_history)


    # ========================================================
    # DISPLAY EAR
    # ========================================================

    cv2.putText(
        frame,
        f"EAR: {smooth_ear:.3f}",
        (10, 100),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )


    # ========================================================
    # EYE STATUS
    # ========================================================

    if smooth_ear < EYE_THRESH:

        # -----------------------------------------------
        # Eyes are closed
        # -----------------------------------------------

        eye_counter += 1

        cv2.putText(
            frame,
            "EYES CLOSED",
            (10, 130),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 0, 255),
            2
        )


    else:

        # -----------------------------------------------
        # Eyes are open
        # -----------------------------------------------

        eye_counter = 0

        # Stop drowsiness sound immediately
        mixer.music.stop()

        # Allow a new alert next time eyes close
        drowsy_alert_played = False

        cv2.putText(
            frame,
            "EYES OPEN",
            (10, 130),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )


    # ========================================================
    # DROWSINESS ALERT
    # ========================================================

    if eye_counter >= EYE_CONSEC_FRAMES:

        cv2.putText(
            frame,
            "DROWSY ALERT!",
            (10, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 0, 255),
            3
        )


        # Play sound only once
        if not drowsy_alert_played:

            mixer.music.play()

            drowsy_alert_played = True


    # ========================================================
    # MOUTH DETECTION
    # ========================================================

    mouth = shape[mouth_start:mouth_end]


    # Calculate MAR
    mar_value = mar(mouth)


    # ========================================================
    # DISPLAY MAR
    # ========================================================

    cv2.putText(
        frame,
        f"MAR: {mar_value:.3f}",
        (10, 160),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 0),
        2
    )


    # ========================================================
    # YAWN DETECTION
    # ========================================================

    if mar_value > MOUTH_THRESH:

        mouth_counter += 1

    else:

        mouth_counter = 0

        # Allow another yawn alert later
        yawn_alert_played = False


    # ========================================================
    # YAWN ALERT
    # ========================================================

    if mouth_counter >= MOUTH_CONSEC_FRAMES:

        cv2.putText(
            frame,
            "YAWN ALERT!",
            (10, 75),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (255, 0, 0),
            3
        )


        # Play sound once
        if not yawn_alert_played:

            # Only start sound if nothing is playing
            if not mixer.music.get_busy():

                mixer.music.play()

            yawn_alert_played = True


    # ========================================================
    # DRAW LEFT EYE
    # ========================================================

    cv2.drawContours(
        frame,
        [cv2.convexHull(left_eye)],
        -1,
        (0, 255, 0),
        2
    )


    # ========================================================
    # DRAW RIGHT EYE
    # ========================================================

    cv2.drawContours(
        frame,
        [cv2.convexHull(right_eye)],
        -1,
        (0, 255, 0),
        2
    )


    # ========================================================
    # DRAW MOUTH
    # ========================================================

    cv2.drawContours(
        frame,
        [cv2.convexHull(mouth)],
        -1,
        (0, 255, 255),
        2
    )


    # ========================================================
    # SHOW CAMERA
    # ========================================================

    cv2.imshow(
        "Drowsiness & Yawn Detection",
        frame
    )


    # ========================================================
    # QUIT
    # ========================================================

    if cv2.waitKey(1) & 0xFF == ord("q"):

        break


# ============================================================
# CLEANUP
# ============================================================

cap.release()

cv2.destroyAllWindows()

mixer.quit()

print("========================================")
print("PROGRAM STOPPED")
print("========================================")

DROWSINESS & YAWN DETECTION STARTED
Press Q to quit.
PROGRAM STOPPED


In [8]:
import os
print(os.getcwd())

C:\Users\polep
